# Extract a Poster Frame from a Video

Pulls a single frame out of a video (e.g. `Showreel.mp4`) and saves it as a JPG.
Use the saved image as the `poster="..."` attribute on an HTML5 `<video>` tag so the
browser has something to paint instantly, before the video itself has loaded.

Requires: `pip install opencv-python`

In [1]:
import cv2
import os

# ---- CONFIG: edit these paths for your project ----
VIDEO_PATH = "static/video/Background Video.mp4"
OUTPUT_PATH = "static/frames/Showreel-poster.jpg"
TIMESTAMP_SECONDS = 1.5   # which point in the video to grab the frame from
JPEG_QUALITY = 90         # 0-100, higher = better quality / larger file

In [2]:
def extract_frame(video_path, output_path, timestamp_seconds=1.5, jpeg_quality=90):
    """Extract a single frame from a video at the given timestamp and save it as a JPG."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 24
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    duration = frame_count / fps if fps else 0

    # Clamp the requested timestamp to the video's actual duration
    ts = max(0, min(timestamp_seconds, max(duration - 0.1, 0)))

    cap.set(cv2.CAP_PROP_POS_MSEC, ts * 1000)
    success, frame = cap.read()
    cap.release()

    if not success or frame is None:
        raise RuntimeError(f"Failed to read a frame at {ts:.2f}s from {video_path}")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    cv2.imwrite(output_path, frame, [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
    print(f"Saved frame at {ts:.2f}s -> {output_path}")
    print(f"Video info: {frame.shape[1]}x{frame.shape[0]}, fps={fps:.2f}, duration={duration:.2f}s")
    return output_path

In [3]:
extract_frame(VIDEO_PATH, OUTPUT_PATH, TIMESTAMP_SECONDS, JPEG_QUALITY)

Saved frame at 1.50s -> static/frames/Showreel-poster.jpg
Video info: 1924x1076, fps=24.00, duration=10.04s


'static/frames/Showreel-poster.jpg'

## Optional: extract several candidate frames

Handy when you want to eyeball a few options and pick the best-looking poster frame
instead of guessing a timestamp.

In [ ]:
def extract_frame_grid(video_path, output_dir, num_frames=6, jpeg_quality=90):
    """Extract `num_frames` evenly spaced frames across the whole video for comparison."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 24
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    duration = frame_count / fps if fps else 0

    os.makedirs(output_dir, exist_ok=True)
    saved_paths = []

    for i in range(num_frames):
        ts = duration * (i + 1) / (num_frames + 1)  # skip the very first/last edges
        cap.set(cv2.CAP_PROP_POS_MSEC, ts * 1000)
        success, frame = cap.read()
        if not success:
            continue
        out_path = os.path.join(output_dir, f"candidate_{i}_{ts:.2f}s.jpg")
        cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
        saved_paths.append(out_path)

    cap.release()
    print(f"Saved {len(saved_paths)} candidate frames to {output_dir}")
    return saved_paths

# Example:
# extract_frame_grid(VIDEO_PATH, "Portfolio/static/video/poster_candidates", num_frames=6)